# 01 — Khám phá dữ liệu (EDA)

Mục tiêu:
- Hiểu cấu trúc dữ liệu HR Analytics
- Phát hiện các yếu tố ảnh hưởng nghỉ việc
- Xác định features quan trọng từ đầu
- Kết nối insight EDA → lựa chọn feature → mô hình

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import load_data, load_config
from src.visualization.plots import (
    plot_attrition_distribution, plot_attrition_by_factor,
    plot_correlation_heatmap, plot_numeric_distributions,
)

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1.1 Tải dữ liệu

In [ ]:
config = load_config("configs/params.yaml")
df = load_data(config["data"]["raw_path"])
print(f"Shape: {df.shape}")
df.head()

## 1.2 Thông tin tổng quan

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Kiểm tra giá trị thiếu
missing = df.isnull().sum()
missing[missing > 0]

## 1.3 Phân bố Attrition (Target)

> **Insight quan trọng**: Dataset mất cân bằng (~16% nghỉ việc) → cần dùng PR-AUC thay vì accuracy, và class_weight='balanced' cho mô hình.

In [ ]:
# Phân bố target
print(df["Attrition"].value_counts())
attrition_rate = (df["Attrition"] == "Yes").mean()
print(f"\nTỷ lệ nghỉ việc: {attrition_rate:.1%}")
plot_attrition_distribution(df)

## 1.4 Phân tích Attrition theo các yếu tố

### Insight chính:
- **OverTime = Yes** → tỷ lệ nghỉ việc **cao gấp 3 lần**
- **Single** → nghỉ việc nhiều hơn Married/Divorced
- **Travel_Frequently** → rủi ro nghỉ việc cao
- **Sales Representative** → tỷ lệ nghỉ việc cao nhất
- **Junior (< 5 năm KN)** → dễ chuyển việc

In [ ]:
# Attrition theo OverTime
plot_attrition_by_factor(df, "OverTime")

# Tính tỷ lệ cụ thể
print("\nTỷ lệ nghỉ việc theo OverTime:")
print(pd.crosstab(df["OverTime"], df["Attrition"], normalize="index").round(3))

In [ ]:
# Attrition theo MaritalStatus
plot_attrition_by_factor(df, "MaritalStatus")

In [ ]:
# Attrition theo Department
plot_attrition_by_factor(df, "Department")

In [ ]:
# Attrition theo BusinessTravel
plot_attrition_by_factor(df, "BusinessTravel")

In [ ]:
# Attrition theo JobRole
plot_attrition_by_factor(df, "JobRole")

## 1.5 Phân bố biến số theo Attrition

> **Insight**: Nhân viên nghỉ việc có xu hướng: tuổi trẻ hơn, thu nhập thấp hơn, ít năm kinh nghiệm hơn, JobSatisfaction thấp hơn.

In [ ]:
plot_numeric_distributions(df)

## 1.6 Ma trận tương quan

In [ ]:
plot_correlation_heatmap(df)

# Top tương quan với target
df_temp = df.copy()
df_temp["Attrition_num"] = df_temp["Attrition"].map({"Yes": 1, "No": 0})
corr_target = df_temp.select_dtypes(include=[np.number]).corr()["Attrition_num"].abs().sort_values(ascending=False)
print("\nTop 10 features tương quan với Attrition:")
print(corr_target.head(11))

## 1.7 Tóm tắt EDA → Hướng đi cho mô hình

| Phát hiện | Ý nghĩa | Hành động |
|---|---|---|
| OT cao → nghỉ việc | Workload quá tải | Tạo WorkloadScore feature |
| Single nghỉ nhiều | Ít gắn bó | Xem xét engagement strategy |
| Junior dễ nghỉ | Career path chưa rõ | Tạo CareerGrowthRate feature |
| Thu nhập thấp → nghỉ | Lương không cạnh tranh | Tạo IncomePerLevel feature |
| Satisfaction quan trọng | Hài lòng tổng hợp | Tạo SatisfactionIndex feature |
| Dataset mất cân bằng (16%) | Accuracy thiên vị | Dùng PR-AUC, class_weight |

→ Các features phái sinh này sẽ được tạo trong notebook 02.